# Phase 1: Continuous Pre-training (CPT) — Llama 3.1 8B on Tamil Wikipedia

**Goal**: Reinforce Tamil language modeling on top of Llama 3.1 8B using 532K Tamil Wikipedia chunks.

**Hardware**: Colab 40GB for full run | Local 8GB: use `LOCAL_TEST = True` with 3B model

**Output**: `wickkiey/tamil-llama-3.1-8b-cpt-v1` on HuggingFace

**Estimated time**: ~4–6 hours on A100 40GB (2 epochs, 532K chunks)

In [1]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Set LOCAL_TEST = True to run a small smoke-test on 8GB VRAM using Llama 3.2 3B
# Set LOCAL_TEST = False for full CPT run on Colab 40GB with Llama 3.1 8B
LOCAL_TEST = True

MODEL_NAME   = "unsloth/Llama-3.2-3B" if LOCAL_TEST else "unsloth/Meta-Llama-3.1-8B"
MAX_SAMPLES  = 300                   if LOCAL_TEST else None   # fewer samples for a fast local smoke test
NUM_EPOCHS   = 1                      if LOCAL_TEST else 2
MAX_SEQ_LEN  = 1024                   if LOCAL_TEST else 2048
BATCH_SIZE   = 2                      if LOCAL_TEST else 4
GRAD_ACCUM   = 4                      if LOCAL_TEST else 8

HF_DATASET   = "wickkiey/tamil-wikipedia-markdown"  # chunked version on HF
# Alternatively, use local file:
# HF_DATASET = None  # set to None to use local file
LOCAL_DATA   = "../data/tawiki_chunked.jsonl"        # local fallback

OUTPUT_DIR   = "outputs/cpt_v1"
HF_REPO      = "wickkiey/tamil-llama-3.1-8b-cpt-v1"

expected_steps = max(1, MAX_SAMPLES // (BATCH_SIZE * GRAD_ACCUM)) if MAX_SAMPLES else "full dataset"

print(f"Model : {MODEL_NAME}")
print(f"Mode  : {'LOCAL TEST (3B, 300 samples)' if LOCAL_TEST else 'FULL CPT (8B, all data)'}")
print(f"Expected steps per epoch: {expected_steps}")


Model : unsloth/Llama-3.2-3B
Mode  : LOCAL TEST (3B, 300 samples)
Expected steps per epoch: 37


In [2]:
# ── Install dependencies ───────────────────────────────────────────────────────
# Run once per session
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install("unsloth")
install("xformers")
install("trl")
install("peft")
install("accelerate")
install("bitsandbytes")
install("datasets")

print("Dependencies installed.")

Dependencies installed.


In [3]:
# ── Load model & tokenizer ─────────────────────────────────────────────────────
import os
import sys
import subprocess
# CPT mode: disables CCE path not supported for continued pretraining.
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"

# On native Windows, a partial vLLM install can break `import unsloth`.
# Unsloth does not require vLLM for CPT training, so we remove it if present.
if sys.platform.startswith("win"):
    _ = subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "vllm"], check=False)

from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,       # auto-detect: bfloat16 on Ampere+, float16 otherwise
    load_in_4bit   = True,       # NF4 quantization for memory efficiency
)

print(f"Loaded: {MODEL_NAME}")
print(f"GPU memory after load: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
ERROR 07-29 03:24:06 [gpt_oss_triton_kernels_moe.py:34] Failed to import Triton kernels. Please make sure your triton version is compatible. Error: No module named 'triton_kernels.routing'
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.9: Fast Llama patching. Transformers: 4.57.6. vLLM: 0.16.1.dev0+g89a77b108.d20260417.cu128.
   \\   /|    NVIDIA GeForce RTX 3070 Ti Laptop GPU. Num GPUs = 1. Max memory: 8.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loaded: unsloth/Llama-3.2-3B
GPU memory after load: 2.4 GB


In [4]:
# ── Apply LoRA for CPT ─────────────────────────────────────────────────────────
# For CPT we use a moderate rank (64) targeting all projection layers.
# This preserves general knowledge while allowing Tamil specialization.
model = FastLanguageModel.get_peft_model(
    model,
    r              = 32 if LOCAL_TEST else 64,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention
        "gate_proj", "up_proj", "down_proj",        # feed-forward
        "embed_tokens", "lm_head",                    # recommended for CPT
    ],
    lora_alpha          = 64 if LOCAL_TEST else 128,
    lora_dropout        = 0,          # 0 is optimal per Unsloth benchmarks
    bias                = "none",
    use_gradient_checkpointing = "unsloth",  # saves ~30% VRAM
    random_state        = 42,
    use_rslora          = True,
)

model.print_trainable_parameters()

Unsloth: Offloading input_embeddings to disk to save VRAM


/opt/venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
/opt/venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:919: UserWarning: Model with `tie_word_embeddings=True` and the tied_target_modules=['lm_head'] are part of the adapter. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. See for example https://github.com/huggingface/peft/issues/2018.
  warnings.warn(
Unsloth 2026.5.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: Training embed_tokens in mixed precision to save VRAM
trainable params: 446,832,640 || all params: 4,053,584,896 || trainable%: 11.0231


In [5]:
# ── Load Tamil Wikipedia dataset ───────────────────────────────────────────────
from datasets import load_dataset
import os

if HF_DATASET:
    print(f"Loading from HuggingFace: {HF_DATASET}")
    # Try chunked version first, fall back to pages
    try:
        dataset = load_dataset(HF_DATASET, split="train")
    except Exception:
        dataset = load_dataset(HF_DATASET, "chunked", split="train")
else:
    print(f"Loading from local file: {LOCAL_DATA}")
    dataset = load_dataset("json", data_files=LOCAL_DATA, split="train")

if MAX_SAMPLES:
    dataset = dataset.select(range(MAX_SAMPLES))
    print(f"Subset selected: {MAX_SAMPLES} samples (LOCAL_TEST mode)")

print(f"Dataset size : {len(dataset):,} samples")
print(f"Columns      : {dataset.column_names}")
print(f"\nSample (first 200 chars):")
print(dataset[0]["text"][:200])

Loading from HuggingFace: wickkiey/tamil-wikipedia-markdown
Subset selected: 300 samples (LOCAL_TEST mode)
Dataset size : 300 samples
Columns      : ['text']

Sample (first 200 chars):
# விக்கிப்பீடியா:கலந்துரையாடல்

***இங்கு தமிழ் விக்கிப்பீடியாவைப் பற்றிய உங்கள் பொதுவான கருத்துக்கள், பாராட்டுக்கள் மற்றும் ஆலோசனைகளைத் தெரிவிக்கலாம். உங்கள் கருத்துகளுக்கு மற்ற விக்கிப்பீடியர்கள் பதி


In [6]:
# ── Format: append EOS token ───────────────────────────────────────────────────
# CPT uses raw text — no chat template, just text + EOS so the model learns
# to complete Tamil sequences and stop at natural boundaries.
EOS_TOKEN = tokenizer.eos_token

def format_cpt(examples):
    # Keep only text column; append EOS
    return {"text": [t + EOS_TOKEN for t in examples["text"]]}

dataset = dataset.map(format_cpt, batched=True, desc="Appending EOS tokens")

# Verify
sample = dataset[0]["text"]
print(f"Sample ends with EOS: {sample.endswith(EOS_TOKEN)}")
print(f"Sample tail: ...{repr(sample[-30:])}")

Appending EOS tokens:   0%|          | 0/300 [00:00<?, ? examples/s]

Sample ends with EOS: True
Sample tail: ...'ணன் (பேச்சு)  \n<|end_of_text|>'


In [7]:
# ── Training arguments ─────────────────────────────────────────────────────────
from unsloth import is_bfloat16_supported, UnslothTrainingArguments

# For local runs, reduce the number of optimizer steps to keep runtime manageable
if LOCAL_TEST:
    logging_steps = 5
    save_steps = 50
    max_steps = 30
else:
    logging_steps = 10
    save_steps = 200
    max_steps = -1

training_args = UnslothTrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,     # effective batch = BATCH * GRAD_ACCUM
    warmup_ratio                = 0.05,
    learning_rate               = 5e-5,
    embedding_learning_rate     = 5e-6,           # 10x smaller for stable CPT
    fp16                        = not is_bfloat16_supported(),
    bf16                        = is_bfloat16_supported(),
    logging_steps               = logging_steps,
    optim                       = "adamw_8bit",   # 8-bit Adam saves ~6GB
    weight_decay                = 0.01,
    lr_scheduler_type           = "cosine",
    seed                        = 42,
    save_strategy               = "steps",
    save_steps                  = save_steps,
    save_total_limit            = 2,              # keep only 2 most recent checkpoints
    report_to                   = "none",         # change to "wandb" if tracking
    max_steps                   = max_steps,
)

print(f"Effective batch size : {BATCH_SIZE * GRAD_ACCUM}")
print(f"Epochs               : {NUM_EPOCHS}")
print(f"BF16 available       : {is_bfloat16_supported()}")
print(f"Max steps            : {max_steps if max_steps != -1 else 'unlimited'}")


Effective batch size : 8
Epochs               : 1
BF16 available       : True
Max steps            : 30


In [8]:
# ── Trainer ────────────────────────────────────────────────────────────────────
# Using UnslothTrainer with packing=True for CPT:
# - packing concatenates short texts to fill MAX_SEQ_LEN → fewer wasted tokens
# - dataset_text_field="text" tells trainer to use the text column directly
from unsloth import UnslothTrainer

trainer = UnslothTrainer(
    model                    = model,
    tokenizer                = tokenizer,
    train_dataset            = dataset,
    dataset_text_field       = "text",
    max_seq_length           = MAX_SEQ_LEN,
    dataset_num_proc         = 4,
    packing                  = True,     # essential for CPT efficiency
    args                     = training_args,
)

# Show memory before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}  |  Total VRAM: {max_memory} GB  |  Reserved: {start_gpu_memory} GB")

Unsloth: Tokenizing ["text"] (num_proc=9):   0%|          | 0/300 [00:00<?, ? examples/s]

GPU: NVIDIA GeForce RTX 3070 Ti Laptop GPU  |  Total VRAM: 8.0 GB  |  Reserved: 3.82 GB


In [9]:
# ── Train ──────────────────────────────────────────────────────────────────────
trainer_stats = trainer.train()

# Memory report
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
print(f"\nPeak VRAM usage : {used_memory} GB")
print(f"VRAM for LoRA   : {used_memory_for_lora} GB")
print(f"Training time   : {trainer_stats.metrics['train_runtime']:.0f} sec")
print(f"Tokens/sec      : {trainer_stats.metrics['train_samples_per_second']:.1f}")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 300 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 446,832,640 of 4,053,584,896 (11.02% trained)


Unsloth: Setting lr = 5.00e-06 instead of 5.00e-05 for embed_tokens.
Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,0.642500
10,0.631000
15,0.624900
20,0.611100
25,0.598300
30,0.582000


/opt/venv/lib/python3.12/site-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")



Peak VRAM usage : 7.283 GB
VRAM for LoRA   : 3.463 GB
Training time   : 351 sec
Tokens/sec      : 0.7


In [10]:
# ── Quick sanity check: generate Tamil text ────────────────────────────────────
FastLanguageModel.for_inference(model)

prompt = "தமிழ்நாடு என்பது"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens  = 100,
    temperature     = 0.7,
    do_sample       = True,
    pad_token_id    = tokenizer.eos_token_id,
)

print("Prompt :", prompt)
print("Output :", tokenizer.decode(outputs[0], skip_special_tokens=True))

Prompt : தமிழ்நாடு என்பது
Output : தமிழ்நாடு என்பது இந்தியாவின் தமிழ்நாடு மாநிலத்தின் தலைநகராகும். இது தமிழ்நாடு மாநிலத்தின் த�


In [11]:
# ── Save LoRA adapter locally ──────────────────────────────────────────────────
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter saved to: {OUTPUT_DIR}")

/opt/venv/lib/python3.12/site-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


LoRA adapter saved to: outputs/cpt_v1


In [12]:
# ── Save to Google Drive (Colab only) ─────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_path = f"/content/drive/MyDrive/Tamil-LLM/{OUTPUT_DIR}"
    model.save_pretrained(drive_path)
    tokenizer.save_pretrained(drive_path)
    print(f"Saved to Google Drive: {drive_path}")
except ImportError:
    print("Not running on Colab — skipping Google Drive save.")

Not running on Colab — skipping Google Drive save.


In [13]:
# ── Push to HuggingFace ────────────────────────────────────────────────────────
# Option A: Push LoRA adapter only (small, fast, recommended)
# Option B: Push merged 16-bit model (large but self-contained)

# Set your HF token here OR use `huggingface-cli login` before running
HF_TOKEN = None  # e.g. "hf_xxxxxxxxxxxx"  — leave None to use cached login

PUSH_TO_HF = not LOCAL_TEST  # don't push during local test runs

if PUSH_TO_HF:
    print("Pushing LoRA adapter to HuggingFace...")
    model.push_to_hub(
        HF_REPO,
        token=HF_TOKEN,
        commit_message="CPT v1 — Llama 3.1 8B on Tamil Wikipedia (532K chunks, 2 epochs)"
    )
    tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
    print(f"Pushed to: https://huggingface.co/{HF_REPO}")
else:
    print("LOCAL_TEST mode — skipping HF push.")

LOCAL_TEST mode — skipping HF push.


In [14]:
# ── (Optional) Push merged 16-bit model ───────────────────────────────────────
# Use this if you need a self-contained model (e.g. for vLLM inference).
# Warning: merged model is ~16GB, takes longer to upload.

PUSH_MERGED = False  # set True only if you need the merged model

if PUSH_MERGED and PUSH_TO_HF:
    print("Pushing merged 16-bit model (this will take a while)...")
    model.push_to_hub_merged(
        HF_REPO + "-merged",
        tokenizer,
        save_method = "merged_16bit",
        token       = HF_TOKEN,
    )
    print(f"Merged model pushed to: https://huggingface.co/{HF_REPO}-merged")

## Next Steps

1. Run `evaluation/03_benchmark_eval.ipynb` on this CPT checkpoint to establish perplexity baseline
2. *Optional A/B*: Repeat this notebook with `MODEL_NAME = "unsloth/Qwen2.5-7B"` — compare perplexity scores; whichever is lower becomes the main track
3. Run `continual-pretraining/02_data_expansion_pipeline.ipynb` to build the expanded corpus
4. Proceed to `finetuning/02_sft_instruction_tuning.ipynb` using this checkpoint